# Using DaoXE (OpenAI-compatible) with Strands Agents

## Overview

[DaoXE](https://daoxe.com) is a multi-model multi-protocol AI API gateway. It exposes an OpenAI-compatible Chat Completions endpoint at `https://daoxe.com/v1`, so Strands can use the built-in `OpenAIModel` provider without a custom package.

This notebook builds a small agent with two demo tools (`current_time`, `current_weather`) routed through DaoXE.

> **Notes**
> - Model IDs are **account-scoped** — use your DaoXE dashboard or `GET https://daoxe.com/v1/models`.
> - DaoXE also supports other protocol surfaces (e.g. Anthropic Messages) for clients that speak those APIs natively; this sample covers the **OpenAI-compatible** path only.
> - DaoXE is **not available in mainland China**.

## Agent Details

| Feature | Description |
|---------|-------------|
| Feature used | OpenAI-compatible model provider (`OpenAIModel`) |
| Agent structure | Single agent |
| Gateway | `https://daoxe.com/v1` |


## Setup and prerequisites

### Prerequisites
* Python 3.10+
* DaoXE API key from [daoxe.com](https://daoxe.com)
* Account-scoped model ID enabled for your key
* Network access outside mainland China

Install packages:


In [ ]:
!pip install -r requirements.txt


### Import dependencies


In [ ]:
import os
from datetime import datetime
from datetime import timezone as tz
from typing import Any
from zoneinfo import ZoneInfo

from strands import Agent, tool
from strands.models.openai import OpenAIModel


### Configure DaoXE credentials

Set `DAOXE_API_KEY` and `DAOXE_MODEL_ID` in your environment (preferred), or assign them in the next cell for a local notebook session.


In [ ]:
# Prefer environment variables in real use:
# export DAOXE_API_KEY="your-daoxe-api-key"
# export DAOXE_MODEL_ID="your-account-model-id"

api_key = os.environ.get("DAOXE_API_KEY", "")
model_id = os.environ.get("DAOXE_MODEL_ID", "")

if not api_key or not model_id:
    raise ValueError(
        "Set DAOXE_API_KEY and DAOXE_MODEL_ID. Model IDs are account-scoped "
        "(dashboard or GET https://daoxe.com/v1/models). DaoXE is not available in mainland China."
    )


### Define demo tools


In [ ]:
@tool
def current_time(timezone: str = "UTC") -> str:
    if timezone.upper() == "UTC":
        timezone_obj: Any = tz.utc
    else:
        timezone_obj = ZoneInfo(timezone)
    return datetime.now(timezone_obj).isoformat()


@tool
def current_weather(city: str) -> str:
    # Dummy implementation. Replace with a real weather API if needed.
    return f"sunny in {city}" 


### Create OpenAIModel pointed at DaoXE

`base_url` must be exactly `https://daoxe.com/v1` (include `/v1`).


In [ ]:
model = OpenAIModel(
    client_args={
        "api_key": api_key,
        "base_url": "https://daoxe.com/v1",
    },
    # Account-scoped model ID from your DaoXE catalog — not a fixed public list
    model_id=model_id,
    params={
        "max_tokens": 2048,
        "temperature": 0.2,
    },
)


### Build the agent


In [ ]:
system_prompt = "You are a simple agent that can tell the time and the weather"
agent = Agent(
    model=model,
    system_prompt=system_prompt,
    tools=[current_time, current_weather],
)


### Invoke the agent


In [ ]:
results = agent("What time is it in Seattle? And how is the weather?")


### Inspect conversation and metrics


In [ ]:
agent.messages


In [ ]:
results.metrics


## Multi-protocol note

This sample uses DaoXE's **OpenAI-compatible** surface. For Anthropic Messages (or other) protocol paths, use the matching client/SDK against the endpoint documented in your DaoXE console. Client setup samples: [DaoXE-AI](https://github.com/seven7763/DaoXE-AI).

## Affiliation

Community contribution from a DaoXE maintainer (`seven7763`).
